# Chapter 3: Data Preparation

This notebook documents the complete data preparation pipeline for the cognitive distortion classification task.

## 3.1 Viewing and Selecting Datasets

The following datasets are inspected for potential inclusion in the final classification corpus:

| # | Dataset | Source | Language | Expected size |
|---|---------|--------|----------|---------------|
| 1a | TherapistQA (Shreevastava & Foltz, 2021) | Kaggle | English | 2,530 |
| 1b | Cognitive Distortion (Danthareja, 2024) | Hugging Face | English | 2,530 |
| 2 | Mendeley Indonesian (Suputra et al., 2025) | Mendeley Data | Indonesian / English | 4,992 |
| 3 | Thinking Traps (Sharma et al., 2023) | GitHub | English | 1,077 |
| 4 | Cognitive Reframing (Sharma et al., 2023) | GitHub | English | 600 |

The Russian Cognitive Distortion Dataset (Vaui) is excluded from the primary English corpus and reserved for multilingual experiments (Chapter 4.4).

For each dataset, the inspection covers: column names, sample records, total record count, label distribution, and exact label strings.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

# Datasets file paths
PATH_ROOT        = './datasets/'
PATH_THERAPISTQA = 'kaggle_cognitive_distortion_detection_dataset.csv'
PATH_MENDELEY    = 'COGNITIVE_DISTORTION_DATASET_IN_ENGLISH_COMPLETE.csv'
PATH_TRAPS       = 'thinking_traps.jsonl'
PATH_REFRAMING   = 'reframing_dataset.csv'

In [2]:
def inspect_dataset(df, name, label_col, text_col=None):
    """Print a summary of a dataset."""
    print('=' * 80)
    print(f'DATASET: {name}')
    print('=' * 80)
    print(f'\nShape: {df.shape[0]} rows x {df.shape[1]} columns')
    print(f'Columns: {list(df.columns)}')
    
    nulls = df.isnull().sum()
    if nulls.any():
        print(f'\nNull values:')
        print(nulls[nulls > 0].to_string())
    else:
        print('\nNo null values.')
    
    print(f'\nLabel distribution (column: "{label_col}"):')
    dist = df[label_col].value_counts()
    for label, count in dist.items():
        print(f'  {str(label):45s} {count:5d}  ({count/len(df)*100:5.1f}%)')
    print(f'  {"TOTAL":45s} {len(df):5d}')
    
    if text_col and text_col in df.columns:
        lengths = df[text_col].astype(str).str.len()
        print(f'\nText length stats (column: "{text_col}"):')
        print(f'  Mean: {lengths.mean():.0f} chars | Median: {lengths.median():.0f} | '
              f'Min: {lengths.min()} | Max: {lengths.max()}')
    
    print(f'\nSample rows (first 3):')
    display(df.head(3))
    print()

---
### Dataset 1a: TherapistQA (Shreevastava & Foltz, 2021)

Source: Kaggle. Contains patient questions from an online therapy Q&A forum, annotated by licensed therapists with a dominant distortion label from the Burns taxonomy.

In [3]:
df_therapistqa = pd.read_csv(PATH_ROOT + PATH_THERAPISTQA)
inspect_dataset(df_therapistqa, 'TherapistQA (Shreevastava & Foltz, 2021)', 'Dominant Distortion', 'Patient Question')

# Distorted part length stats
dp = df_therapistqa['Distorted part'].dropna()
lengths = dp.astype(str).str.len()
print(f'Distorted part length stats: Mean: {lengths.mean():.0f} | Median: {lengths.median():.0f} | Min: {lengths.min()} | Max: {lengths.max()}')

# Secondary distortion coverage
sec = df_therapistqa['Secondary Distortion (Optional)'].notna().sum()
print(f'Records with secondary distortion: {sec}/{len(df_therapistqa)} ({sec/len(df_therapistqa)*100:.1f}%)')

DATASET: TherapistQA (Shreevastava & Foltz, 2021)

Shape: 2530 rows x 5 columns
Columns: ['Id_Number', 'Patient Question', 'Distorted part', 'Dominant Distortion', 'Secondary Distortion (Optional)']

Null values:
Distorted part                      933
Secondary Distortion (Optional)    2114

Label distribution (column: "Dominant Distortion"):
  No Distortion                                   933  ( 36.9%)
  Mind Reading                                    239  (  9.4%)
  Overgeneralization                              239  (  9.4%)
  Magnification                                   195  (  7.7%)
  Labeling                                        165  (  6.5%)
  Personalization                                 153  (  6.0%)
  Fortune-telling                                 143  (  5.7%)
  Emotional Reasoning                             134  (  5.3%)
  Mental filter                                   122  (  4.8%)
  Should statements                               107  (  4.2%)
  All-or-nothi

,Id_Number,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,4500,"Hello, I have a beautiful,smart,outgoing and amazing five year old little girl. Yesterday she came to me and said mo...",The voice are always fimilar (someone she knows) Im very concerned about this and hope it has nothing to do with my ...,Personalization,NaN
1,4501,Since I was about 16 years old I’ve had these “spells” where it’s impossible for me to control my emotions. They com...,I feel trapped inside my disgusting self and like I’m willing to do anything to escape it.I’m just afraid sometimes ...,Labeling,Emotional Reasoning
2,4502,So I’ve been dating on and off this guy for almost 3 years. He is 3 years younger than me. During our relationship ...,NaN,No Distortion,NaN



Distorted part length stats: Mean: 189 | Median: 151 | Min: 11 | Max: 1824
Records with secondary distortion: 416/2530 (16.4%)


---
### Dataset 1b: Cognitive Distortion (Danthareja, 2024)

Source: Hugging Face. Should have the same record structure as TherapistQA and a pre-made train/test split. The relationship between this dataset and TherapistQA must be verified empirically.

In [4]:
# Fetch the danthareja dataset from Hugging Face
from datasets import load_dataset

ds = load_dataset('danthareja/cognitive-distortion')
print(f'Splits: {list(ds.keys())}')
print(f'Train size: {len(ds["train"])}, Test size: {len(ds["test"])}')
print(f'Columns: {ds["train"].column_names}')

# Combine train + test into a single DataFrame
df_danthareja = pd.concat([
    ds['train'].to_pandas(),
    ds['test'].to_pandas()
], ignore_index=True)

print(f'\nCombined size: {len(df_danthareja)}')
print(f'Columns: {list(df_danthareja.columns)}')

display(df_danthareja.head(3))

# Label distribution
label_col_d = None
for c in df_danthareja.columns:
    if 'distortion' in c.lower() and 'secondary' not in c.lower():
        label_col_d = c
        break
if label_col_d is None:
    label_col_d = df_danthareja.columns[-1]  # fallback

print(f'\nDanthareja label distribution (column: "{label_col_d}"):')
dist = df_danthareja[label_col_d].value_counts()
for label, count in dist.items():
    print(f'  {str(label):45s} {count:5d}  ({count/len(df_danthareja)*100:5.1f}%)')

Splits: ['train', 'test']
Train size: 2024, Test size: 506
Columns: ['id', ' patient_question', 'distorted_part', 'dominant_distortion']

Combined size: 2530
Columns: ['id', ' patient_question', 'distorted_part', 'dominant_distortion']


,id,patient_question,distorted_part,dominant_distortion
0,2488,"I have PTSD, have been in a few abusive relationships, have over come them all and have always been fine. This last ...","I am now 6 months pregnant, lately (past 3 or more weeks) every time I lay down to sleep or am calm, the memory and ...",7
1,453,"From the U.S.: I am 13 years old. At the age of 5, a night of 2010 started a 5-6 year long divorce. Our father was a...","thers, to my friends especially, I appear bright and happy, and they never notice what’s happening to me.",8
2,1959,"Hi..in the past, I’d say about 2 years, on and off again I experience drastic mood swings that can cause me to feel...",I feel like it is not worth the trouble of dealing with and this just drives me deeper in since the lack of work et...,0



Danthareja label distribution (column: "dominant_distortion"):
  2                                               933  ( 36.9%)
  7                                               239  (  9.4%)
  8                                               239  (  9.4%)
  3                                               195  (  7.7%)
  10                                              165  (  6.5%)
  5                                               153  (  6.0%)
  4                                               143  (  5.7%)
  1                                               134  (  5.3%)
  9                                               122  (  4.8%)
  6                                               107  (  4.2%)
  0                                               100  (  4.0%)


---
### Comparison: TherapistQA vs Danthareja

This cell verifies whether the Danthareja dataset is a superset of TherapistQA, and identifies any differences in records, labels, or text content. Danthareja README file contains a label mappings which correspond 10 categories used in TherapistQA dataset.

In [5]:
# Map Danthareja numeric labels to readable names
danthareja_label_map = {
    0: 'All-or-nothing thinking',
    1: 'Emotional Reasoning',
    2: 'No Distortion',
    3: 'Magnification',
    4: 'Fortune-telling',
    5: 'Personalization',
    6: 'Should statements',
    7: 'Overgeneralization',
    8: 'Mind Reading',
    9: 'Mental filter',
    10: 'Labeling',
}

# Apply mapping
label_col_d = 'dominant_distortion'
df_danthareja[label_col_d] = df_danthareja[label_col_d].map(danthareja_label_map)

print('Danthareja labels after mapping:')
print(df_danthareja[label_col_d].value_counts().to_string())

# Identify text columns
text_col_t = 'Patient Question'  # TherapistQA

text_col_d = None
for c in df_danthareja.columns:
    if 'question' in c.lower() or 'patient' in c.lower() or 'text' in c.lower():
        text_col_d = c
        break
if text_col_d is None:
    text_col_d = df_danthareja.select_dtypes(include='object').apply(lambda x: x.str.len().mean()).idxmax()

print(f'\nTherapistQA text column: "{text_col_t}"')
print(f'Danthareja text column:  "{text_col_d}"')

# Normalise texts for comparison
therapistqa_texts = set(df_therapistqa[text_col_t].astype(str).str.strip().str.lower())
danthareja_texts  = set(df_danthareja[text_col_d].astype(str).str.strip().str.lower())

only_in_therapistqa = therapistqa_texts - danthareja_texts
only_in_danthareja  = danthareja_texts - therapistqa_texts
in_both             = therapistqa_texts & danthareja_texts

print(f'\n--- Overlap analysis ---')
print(f'TherapistQA unique texts:  {len(therapistqa_texts)}')
print(f'Danthareja unique texts:   {len(danthareja_texts)}')
print(f'Texts in BOTH:             {len(in_both)}')
print(f'Only in TherapistQA:       {len(only_in_therapistqa)}')
print(f'Only in Danthareja:        {len(only_in_danthareja)}')

if len(only_in_therapistqa) > 0:
    print(f'\nSample texts only in TherapistQA (first 3):')
    for t in list(only_in_therapistqa)[:3]:
        print(f'  "{t[:100]}..."')

if len(only_in_danthareja) > 0:
    print(f'\nSample texts only in Danthareja (first 3):')
    for t in list(only_in_danthareja)[:3]:
        print(f'  "{t[:100]}..."')

# For overlapping texts, check if labels match
if len(in_both) > 0:
    label_col_t = 'Dominant Distortion'

    tqa_lookup = {}
    for _, row in df_therapistqa.iterrows():
        key = str(row[text_col_t]).strip().lower()
        tqa_lookup[key] = str(row[label_col_t]).strip()

    dan_lookup = {}
    for _, row in df_danthareja.iterrows():
        key = str(row[text_col_d]).strip().lower()
        dan_lookup[key] = str(row[label_col_d]).strip()

    mismatches = 0
    mismatch_examples = []
    for text in in_both:
        if tqa_lookup.get(text, '') != dan_lookup.get(text, ''):
            mismatches += 1
            if len(mismatch_examples) < 5:
                mismatch_examples.append((text[:80], tqa_lookup.get(text, '?'), dan_lookup.get(text, '?')))

    print(f'\n--- Label agreement on overlapping texts ---')
    print(f'Matching labels:    {len(in_both) - mismatches}/{len(in_both)}')
    print(f'Mismatched labels:  {mismatches}/{len(in_both)}')

    if mismatch_examples:
        print(f'\nMismatch examples:')
        for text, tqa_label, dan_label in mismatch_examples:
            print(f'  Text: "{text}..."')
            print(f'    TherapistQA: {tqa_label} | Danthareja: {dan_label}')

# Check if Danthareja's "No Distortion" records are the only addition
has_nd = 'No Distortion' in df_danthareja[label_col_d].values
danthareja_nd = df_danthareja[df_danthareja[label_col_d] == 'No Distortion'] if has_nd else pd.DataFrame()
danthareja_distorted = df_danthareja[df_danthareja[label_col_d] != 'No Distortion'] if has_nd else df_danthareja

print(f'\n--- Summary ---')
print(f'Danthareja total:         {len(df_danthareja)}')
print(f'Danthareja distorted:     {len(danthareja_distorted)}')
print(f'Danthareja No Distortion: {len(danthareja_nd)}')
print(f'TherapistQA total:        {len(df_therapistqa)}')

Danthareja labels after mapping:
dominant_distortion
No Distortion              933
Overgeneralization         239
Mind Reading               239
Magnification              195
Labeling                   165
Personalization            153
Fortune-telling            143
Emotional Reasoning        134
Mental filter              122
Should statements          107
All-or-nothing thinking    100

TherapistQA text column: "Patient Question"
Danthareja text column:  " patient_question"

--- Overlap analysis ---
TherapistQA unique texts:  2529
Danthareja unique texts:   2529
Texts in BOTH:             2529
Only in TherapistQA:       0
Only in Danthareja:        0

--- Label agreement on overlapping texts ---
Matching labels:    2528/2529
Mismatched labels:  1/2529

Mismatch examples:
  Text: "i grew up in a upper middle class family. i was given most things i wanted and a..."
    TherapistQA: Overgeneralization | Danthareja: No Distortion

--- Summary ---
Danthareja total:         2530
Danthar

- All 2,529 unique texts are shared, the two datasets are the same data. Danthareja simply re-hosted TherapistQA on Hugging Face with a train/test split and the numeric label encoding. TherapistQA dataset will be used further.
- Upon testing if two datasets match, another conclusion was found. Both datasets have 2,529 unique text values, which means that one text is duplicated with different dominant distortion. The next cell is to find that duplicated record.

In [6]:
# Find the duplicate text within TherapistQA
dup_mask = df_therapistqa['Patient Question'].duplicated(keep=False)
dup_rows = df_therapistqa[dup_mask][['Id_Number', 'Dominant Distortion', 'Patient Question']]
print(f'Duplicate rows in TherapistQA: {len(dup_rows)}')
display(dup_rows)

Duplicate rows in TherapistQA: 2


,Id_Number,Dominant Distortion,Patient Question
464,265,No Distortion,I grew up in a upper middle class family. I was given most things I wanted and all the essential things a child woul...
506,307,Overgeneralization,I grew up in a upper middle class family. I was given most things I wanted and all the essential things a child woul...


The record is classified as a 'No Distortion' and 'Overgeneralization'. Based on the actual Patient Question and the fact that Distorted part was extracted in TherapistQA dataset, a record with 'No Distortion' classification will be discarded.

In [7]:
# Remove the duplicate record (Id_Number == 265) from TherapistQA
print(f'TherapistQA size before removal: {len(df_therapistqa)}')

df_therapistqa = df_therapistqa[df_therapistqa['Id_Number'] != 265]

print(f'TherapistQA size after removal:  {len(df_therapistqa)}')
print(f'Unique texts: {df_therapistqa["Patient Question"].nunique()}')

TherapistQA size before removal: 2530
TherapistQA size after removal:  2529
Unique texts: 2529


---
### Dataset 2: Mendeley Indonesian, English version (Suputra et al., 2025)

Source: Mendeley Data. Originally collected in Bahasa Indonesia via online questionnaire, with English back-translation provided. Uses the Burns (1980) 10-category taxonomy. Four under-represented classes were augmented via back-translation through five languages.

In [8]:
df_mendeley = pd.read_csv(PATH_ROOT + PATH_MENDELEY)
inspect_dataset(df_mendeley, 'Mendeley Indonesian, English (Suputra et al., 2025)', 'FIRST ANNOTATOR', 'TEXT (ENGLISH)')

# Annotator agreement
agree = (df_mendeley['FIRST ANNOTATOR'] == df_mendeley['SECOND ANNOTATOR']).sum()
print(f'Inter-annotator agreement: {agree}/{len(df_mendeley)} ({agree/len(df_mendeley)*100:.1f}%)')
print('NOTE: 100% agreement is unusual -> likely a finalised/reconciled dataset, not independent annotations.')

# Data status breakdown
print(f'\nDATA STATUS breakdown:')
for status, count in df_mendeley['DATA STATUS'].value_counts().items():
    print(f'  {status:15s} {count:5d}  ({count/len(df_mendeley)*100:.1f}%)')

DATASET: Mendeley Indonesian, English (Suputra et al., 2025)

Shape: 4992 rows x 5 columns
Columns: ['TEXT (BAHASA)', 'TEXT (ENGLISH)', 'DATA STATUS', 'FIRST ANNOTATOR', 'SECOND ANNOTATOR']

No null values.

Label distribution (column: "FIRST ANNOTATOR"):
  No Distortion                                  2246  ( 45.0%)
  Labeling                                        390  (  7.8%)
  Mind Reading                                    385  (  7.7%)
  Should statement                                371  (  7.4%)
  Personalization and Blame                       284  (  5.7%)
  Discounting the positives                       241  (  4.8%)
  Overgeneralization                              215  (  4.3%)
  All-or-nothing                                  200  (  4.0%)
  Emotional Reasoning                             200  (  4.0%)
  Magnification or Minimization                   200  (  4.0%)
  Mental filter                                   200  (  4.0%)
  Fortune-telling                       

,TEXT (BAHASA),TEXT (ENGLISH),DATA STATUS,FIRST ANNOTATOR,SECOND ANNOTATOR
0,ketika saya sudah berusaha semaksimal mungkin tapi tetap kurang maksimal karena $saya merasa saya tidak begitu baik ...,when I have tried my best but am still not optimal because I feel I am not very good and push myself too much,RAW-ORI,All-or-nothing,All-or-nothing
1,"Saya ketika SD hingga SMP dulu beranggapan bahwa ketika saya tidak menduduki peringkat 1 di kelas, saya berarti kura...","When I was in elementary school to middle school, I used to think that if I didn't rank 1st in class, it meant I was...",RAW-ORI,All-or-nothing,All-or-nothing
2,Saya berfikir jika tidak lolos MBKM maka masa depan saya akan suram. Untungnya saya lolos MBKM.,I thought that if I didn't pass MBKM then my future would be bleak. Luckily I passed MBKM.,RAW-ORI,All-or-nothing,All-or-nothing



Inter-annotator agreement: 4992/4992 (100.0%)
NOTE: 100% agreement is unusual -> likely a finalised/reconciled dataset, not independent annotations.

DATA STATUS breakdown:
  RAW-ORI          4662  (93.4%)
  DIS-ZH            203  (4.1%)
  DIS-EN             44  (0.9%)
  DIS-JV             35  (0.7%)
  DIS-TG             31  (0.6%)
  DIS-MS             17  (0.3%)


In [9]:
# Quality issues
short = df_mendeley[df_mendeley['TEXT (ENGLISH)'].str.len() < 10]
print(f'\nVery short English texts (<10 chars): {len(short)}')
for _, row in short.iterrows():
    print(f'  \"{row["TEXT (ENGLISH)"]}\" [{row["FIRST ANNOTATOR"]}]')


Very short English texts (<10 chars): 2
  "$" [Mind Reading]
  "$" [Personalization and Blame]


In [10]:
# English text quality assessment: random sample of original vs augmented
print('--- ORIGINAL (RAW-ORI) translation quality sample ---')
ori = df_mendeley[df_mendeley['DATA STATUS'] == 'RAW-ORI'].sample(8, random_state=42)
for _, row in ori.iterrows():
    print(f'\n  [{row["FIRST ANNOTATOR"]}]')
    print(f'  {row["TEXT (ENGLISH)"][:200]}')

print('\n\n--- AUGMENTED translation quality sample ---')
aug = df_mendeley[df_mendeley['DATA STATUS'] != 'RAW-ORI'].sample(5, random_state=42)
for _, row in aug.iterrows():
    print(f'\n  [{row["FIRST ANNOTATOR"]}] ({row["DATA STATUS"]})')
    print(f'  {row["TEXT (ENGLISH)"][:200]}')

--- ORIGINAL (RAW-ORI) translation quality sample ---

  [Mind Reading]
  When a friend answered a message briefly, $I thought that I was just bothering him.$ even though he answered my message while doing other activities so he only had time to reply briefly.

  [No Distortion]
  Information in society that is not necessarily true and I have spread it to other people without first investigating it.

  [Personalization and Blame]
  The business I created with my friend was not running stably. Where currently I feel that this business is not running because of my actions, even though I have done it as best as I can, I still feel 

  [All-or-nothing]
  I just want to be friends with everyone, $if anyone shows intentions other than being friends I choose not to be friends with him.$

  [Mind Reading]
  I often think about $“Someone likes me or hates me”. Without thinking about the various feelings and attitudes someone might have towards me.$

  [No Distortion]
  when asking for first imp

---
### Dataset 3: Thinking Traps (Sharma et al., 2023)

Source: GitHub. Contains records in prompt-completion (JSONL) format, where each prompt is a short thought ending with `->` and the completion is the dominant thinking trap label. Uses 13 distortion categories plus a Not Distorted class.

In [11]:
df_traps = pd.read_json(PATH_ROOT + PATH_TRAPS, lines=True)
df_traps['completion'] = df_traps['completion'].str.strip()

inspect_dataset(df_traps, 'Thinking Traps (Sharma et al., 2023)', 'completion', 'prompt')

# Duplicate analysis
prompts_clean = df_traps['prompt'].str.strip().str.rstrip(' ->')
n_unique = prompts_clean.nunique()
n_dup = len(df_traps) - n_unique
print(f'Unique prompts (after stripping "->"): {n_unique}')
print(f'Rows that are multi-label (same prompt, different labels): {n_dup}')

# Show examples of multi-label
dup_mask = prompts_clean.duplicated(keep=False)
dup_prompts_sorted = df_traps[dup_mask].copy()
dup_prompts_sorted['clean'] = prompts_clean[dup_mask]
print(f'\nExamples of multi-label entries (same thought, different labels):')
for p in dup_prompts_sorted['clean'].unique()[:4]:
    labels = df_traps.loc[prompts_clean == p, 'completion'].tolist()
    print(f'  "{p[:80]}" -> {labels}')

print(f'\nLABEL INCONSISTENCY: Both "Overgeneralizing" (n=80) and "Overgeneralization" (n=35) present.')
print(f'LABEL INCONSISTENCY: Both "Personalizing" (n=64) and "Personalization" (n=16) present.')
print(f'LABEL INCONSISTENCY: Both "Disqualifying the Positive" and "Disqualifying the positive" (case diff).')

DATASET: Thinking Traps (Sharma et al., 2023)

Shape: 1077 rows x 2 columns
Columns: ['prompt', 'completion']

No null values.

Label distribution (column: "completion"):
  Catastrophizing                                 110  ( 10.2%)
  Labeling                                        104  (  9.7%)
  Negative feeling or emotion                     102  (  9.5%)
  All-or-nothing thinking                          97  (  9.0%)
  Fortune telling                                  85  (  7.9%)
  Mind reading                                     81  (  7.5%)
  Overgeneralizing                                 80  (  7.4%)
  Not distorted                                    77  (  7.1%)
  Personalizing                                    64  (  5.9%)
  Should statements                                53  (  4.9%)
  Emotional reasoning                              48  (  4.5%)
  Blaming                                          39  (  3.6%)
  Disqualifying the Positive                       36  (  3.3

,prompt,completion
0,I'm always late ->,Overgeneralization
1,I'm late for the meeting. Everyone will look down upon me ->,Mind reading
2,I'm late for the meeting. Everyone will look down upon me ->,Fortune telling



Unique prompts (after stripping "->"): 698
Rows that are multi-label (same prompt, different labels): 379

Examples of multi-label entries (same thought, different labels):
  "I'm late for the meeting. Everyone will look down upon me" -> ['Mind reading', 'Fortune telling']
  "I'm late for the meeting. I'll make a fool of myself" -> ['Labeling', 'Fortune telling']
  "I'm completely self-centered and thoughtless" -> ['All-or-nothing thinking', 'Labeling']
  "I'll make a fool of myself" -> ['Labeling', 'Fortune telling']

LABEL INCONSISTENCY: Both "Overgeneralizing" (n=80) and "Overgeneralization" (n=35) present.
LABEL INCONSISTENCY: Both "Personalizing" (n=64) and "Personalization" (n=16) present.
LABEL INCONSISTENCY: Both "Disqualifying the Positive" and "Disqualifying the positive" (case diff).


In [12]:
# Investigate "Comparing and Despairing" vs "Comparing" in Thinking Traps
comparing_rows = df_traps[df_traps['completion'].str.lower().str.contains('comparing')]
print(f'Records with "Comparing" in label: {len(comparing_rows)}')
print(comparing_rows['completion'].value_counts().to_string())

print(f'\nAll "Comparing" examples:')
for _, row in comparing_rows.iterrows():
    print(f'  [{row["completion"]}] {row["prompt"].strip().rstrip(" ->")}')

Records with "Comparing" in label: 27
completion
Comparing and Despairing    19
Comparing                    8

All "Comparing" examples:
  [Comparing] The color in their work is so much more vibrant, mine doesn't look this good
  [Comparing] My present was much smaller in size than the others
  [Comparing] My niece's birthday party had twice the amount of people
  [Comparing] My father always prefers my elder brother
  [Comparing] She is more successful than I am
  [Comparing] Others did better than I did on the test
  [Comparing] People my age are more successful than I am
  [Comparing] Because other did better than me
  [Comparing and Despairing] I had a fight with my husband My best friend doesn't fight with my husband this much, so I must be doing something wrong
  [Comparing and Despairing] I saw a profile picture of a close friend on a dating app that I had feelings for. I am not the kind of person that she would like to go out with.
  [Comparing and Despairing] I had been worki

In [13]:
# Handle multi-label in Thinking Traps
# Keep the first label per unique prompt (dominant), store all labels as metadata

prompts_clean = df_traps['prompt'].str.strip().str.rstrip(' ->').str.strip()
df_traps['text_clean'] = prompts_clean
df_traps['completion'] = df_traps['completion'].str.strip()

# Group by unique text, take first label as dominant, collect all labels
traps_grouped = df_traps.groupby('text_clean').agg(
    distortion=('completion', 'first'),
    all_labels=('completion', list),
    n_labels=('completion', 'count')
).reset_index()

traps_grouped = traps_grouped.rename(columns={'text_clean': 'text'})

print(f'Thinking Traps: {len(df_traps)} rows -> {len(traps_grouped)} unique texts')
print(f'Single-label: {(traps_grouped["n_labels"] == 1).sum()}')
print(f'Multi-label:  {(traps_grouped["n_labels"] > 1).sum()}')
print(f'\nLabel distribution (dominant label):')
print(traps_grouped['distortion'].value_counts().to_string())

Thinking Traps: 1077 rows -> 698 unique texts
Single-label: 459
Multi-label:  239

Label distribution (dominant label):
distortion
Labeling                       70
Not distorted                  67
Catastrophizing                64
Mind reading                   62
Fortune telling                54
All-or-nothing thinking        51
Should statements              47
Negative feeling or emotion    46
Emotional reasoning            35
Overgeneralization             34
Overgeneralizing               33
Personalizing                  30
Blaming                        25
Disqualifying the positive     23
Disqualifying the Positive     21
Personalization                16
Comparing and Despairing       12
Comparing                       8


---
### Dataset 4: Cognitive Reframing (Sharma et al., 2023)

Source: GitHub. Primarily designed for cognitive reframing experiments, but the `thought` and `thinking_traps_addressed` columns can be repurposed for classification. Uses the same 13-category taxonomy as the Thinking Traps dataset. Many records are multi-label (comma-separated).

In [14]:
df_reframing = pd.read_csv(PATH_ROOT + PATH_REFRAMING)
print('=' * 80)
print('DATASET: Cognitive Reframing (Sharma et al., 2023)')
print('=' * 80)
print(f'\nShape: {df_reframing.shape[0]} rows x {df_reframing.shape[1]} columns')
print(f'Columns: {list(df_reframing.columns)}')

display(df_reframing.head(3))

# Duplicate structure analysis
n_unique_thoughts = df_reframing['thought'].nunique()
n_unique_pairs = df_reframing.drop_duplicates(subset=['thought', 'thinking_traps_addressed']).shape[0]
print(f'\nTotal rows: {len(df_reframing)}')
print(f'Unique thoughts: {n_unique_thoughts}')
print(f'Unique (thought, label) pairs: {n_unique_pairs}')
print(f'Each thought appears ~{len(df_reframing) / n_unique_thoughts:.1f}x (one per reframe variant)')

# Multi-label analysis
df_unique = df_reframing.drop_duplicates(subset=['thought', 'thinking_traps_addressed'])
multi = df_unique['thinking_traps_addressed'].str.contains(',').sum()
print(f'\nRecords with multiple labels (comma-separated): {multi}/{len(df_unique)} ({multi/len(df_unique)*100:.1f}%)')

# Label distribution on unique (thought, label) pairs only
first_trap = df_unique['thinking_traps_addressed'].str.split(',').str[0].str.strip()
print(f'\nLabel distribution (first/dominant label, unique thoughts only):')
dist = first_trap.value_counts()
for label, count in dist.items():
    print(f'  {label:45s} {count:5d}  ({count/len(df_unique)*100:5.1f}%)')
print(f'  {"TOTAL":45s} {len(df_unique):5d}')

lengths = df_unique['thought'].astype(str).str.len()
print(f'\nThought length stats (unique): Mean: {lengths.mean():.0f} | Median: {lengths.median():.0f} | Min: {lengths.min()} | Max: {lengths.max()}')

# Reframe analysis (for Chapter 5)
print(f'\n{"─"*80}')
print(f'REFRAME ANALYSIS (relevant for Chapter 5)')
print(f'{"─"*80}')

# Reframes per unique thought
reframes_per_thought = df_reframing.groupby('thought')['reframe'].count()
print(f'\nReframes per thought:')
for n, count in reframes_per_thought.value_counts().sort_index().items():
    print(f'  {n} reframes: {count} thoughts')

# Reframe length stats
ref_lengths = df_reframing['reframe'].astype(str).str.len()
print(f'\nReframe length stats: Mean: {ref_lengths.mean():.0f} | Median: {ref_lengths.median():.0f} | Min: {ref_lengths.min()} | Max: {ref_lengths.max()}')

# Compare thought vs reframe length
print(f'\nLength comparison (thought → reframe):')
print(f'  Avg thought: {df_reframing["thought"].str.len().mean():.0f} chars')
print(f'  Avg reframe: {df_reframing["reframe"].str.len().mean():.0f} chars')
print(f'  Reframes are on average {df_reframing["reframe"].str.len().mean() / df_reframing["thought"].str.len().mean():.1f}x longer than thoughts')

# Sample thought, reframe pairs
print(f'\nSample thought → reframe pairs:')
for thought in df_reframing['thought'].unique()[:3]:
    rows = df_reframing[df_reframing['thought'] == thought]
    trap = rows.iloc[0]['thinking_traps_addressed']
    print(f'\n  [{trap}] "{thought}"')
    for _, row in rows.iterrows():
        print(f'    → "{row["reframe"][:100]}..."')

DATASET: Cognitive Reframing (Sharma et al., 2023)

Shape: 600 rows x 4 columns
Columns: ['situation', 'thought', 'reframe', 'thinking_traps_addressed']


,situation,thought,reframe,thinking_traps_addressed
0,A Roomate of mine stole my comptuer,"Someone I trusted stole something valuable of mine, I was extremely angry and wanted justice","My roommate stole something of mine, and I will focus on actionable solutions to address this.",emotional reasoning
1,A Roomate of mine stole my comptuer,"Someone I trusted stole something valuable of mine, I was extremely angry and wanted justice","While I would like there to be consequences for my roommate's actions, it may not always be possible.",emotional reasoning
2,"A few days ago, I got angry at my husband's grandmother who lives next door for beeping her horn till I came out of ...",She doesn't respect me.,She is older and may have been tired,overgeneralizing



Total rows: 600
Unique thoughts: 286
Unique (thought, label) pairs: 474
Each thought appears ~2.1x (one per reframe variant)

Records with multiple labels (comma-separated): 160/474 (33.8%)

Label distribution (first/dominant label, unique thoughts only):
  negative feeling or emotion                      60  ( 12.7%)
  labeling                                         54  ( 11.4%)
  mind reading                                     47  (  9.9%)
  fortune telling                                  47  (  9.9%)
  overgeneralizing                                 46  (  9.7%)
  all-or-nothing thinking                          45  (  9.5%)
  personalizing                                    42  (  8.9%)
  disqualifying the positive                       32  (  6.8%)
  catastrophizing                                  30  (  6.3%)
  emotional reasoning                              23  (  4.9%)
  none                                             16  (  3.4%)
  blaming                              

In [15]:
# Cross-dataset overlap: Thinking Traps vs Reframing (both from Sharma et al.)
traps_texts = set(df_traps['prompt'].str.strip().str.rstrip(' ->').str.strip().str.lower())
reframe_thoughts = set(df_reframing['thought'].str.strip().str.lower())

overlap = traps_texts & reframe_thoughts
print(f'Thinking Traps unique prompts (normalised): {len(traps_texts)}')
print(f'Reframing unique thoughts (normalised): {len(reframe_thoughts)}')
print(f'Overlap between the two datasets: {len(overlap)}')

Thinking Traps unique prompts (normalised): 698
Reframing unique thoughts (normalised): 286
Overlap between the two datasets: 9


---
### Dataset 5: PatternReframe (Maddela et al., 2023)

Source: GitHub. The biggest crowdsourced dataset jet with the richest data format.
The dataset is in JSONL format where each line is a JSON object containing:
- persona: fictional character profile used to condition thought generation
- pattern: the dominant cognitive distortion label
- pattern_def: definition of the distortion pattern
- thought: the unhelpful thought (human-written by crowdsource workers)
- strategies_def: definitions of 5 reframing strategies
- reframes: list of 2-3 reframed versions, each with strategy annotations
- marked_patterns: annotator votes for all 12 distortion categories

In [18]:
import json
import os
import pandas as pd

# ── Load all PatternReframe splits (train / valid / test) ──
pattern_dir = './datasets/'
pattern_files = sorted([f for f in os.listdir(pattern_dir)
                        if 'reframe_thoughts' in f and f.endswith('.txt')])

splits = {}           # {split_name: [records]}
all_records = []

for fname in pattern_files:
    fpath = os.path.join(pattern_dir, fname)
    # Infer split name from filename stem (last underscore-separated token)
    # e.g. "reframe_thoughts_dataset_parlai_train.txt" -> "train"
    split_name = fname.replace('.txt', '').split('_')[-1]
    records = []
    with open(fpath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    splits[split_name] = records
    all_records.extend(records)
    print(f'  {fname}: {len(records)} records  (split="{split_name}")')

print(f'\nTotal records across all splits: {len(all_records)}')

# ── Parse into DataFrame (with split column) ──
parsed = []
for split_name, records in splits.items():
    for r in records:
        marked = r.get('marked_patterns', {})
        dominant_votes = int(marked.get(r['pattern'], '0'))
        total_votes = sum(int(v) for v in marked.values())

        parsed.append({
            'split': split_name,
            'thought': r['thought'],
            'pattern': r['pattern'],
            'persona': r['persona'],
            'n_reframes': len(r.get('reframes', [])),
            'dominant_votes': dominant_votes,
            'total_votes': total_votes,
        })

df_pattern = pd.DataFrame(parsed)

print(f'\n{"="*80}')
print(f'DATASET: PatternReframe (Maddela et al., 2023)')
print(f'{"="*80}')
print(f'\nShape: {len(df_pattern)} thoughts, {df_pattern["n_reframes"].sum()} total reframes')
print(f'Unique thoughts: {df_pattern["thought"].nunique()}')
print(f'Unique personas: {df_pattern["persona"].nunique()}')
print(f'Splits loaded: {", ".join(f"{k} ({len(v)})" for k, v in splits.items())}')

# ── Per-split breakdown (thoughts + reframes) ──
print(f'\nPer-split breakdown:')
print(f'  {"split":10s} {"thoughts":>10s} {"reframes":>10s} {"avg rfr/tht":>12s}')
for s in splits.keys():
    sub = df_pattern[df_pattern['split'] == s]
    n_tht = len(sub)
    n_rfr = sub['n_reframes'].sum()
    avg = n_rfr / n_tht if n_tht else 0
    print(f'  {s:10s} {n_tht:>10d} {n_rfr:>10d} {avg:>12.2f}')

# ── Label distribution (overall) ──
print(f'\nLabel distribution ("pattern") — overall:')
dist = df_pattern['pattern'].value_counts()
for label, count in dist.items():
    print(f'  {label:55s} {count:5d}  ({count/len(df_pattern)*100:5.1f}%)')
print(f'  {"TOTAL":55s} {len(df_pattern):5d}')

# ── Label distribution per split (cross-tab) ──
print(f'\nLabel distribution per split:')
ctab = pd.crosstab(df_pattern['pattern'], df_pattern['split'], margins=True, margins_name='TOTAL')
# reorder columns so TOTAL is last
split_order = [s for s in ['train', 'valid', 'test'] if s in ctab.columns] + ['TOTAL']
ctab = ctab[[c for c in split_order if c in ctab.columns]]
print(ctab.to_string())

# ── Text length stats ──
lengths = df_pattern['thought'].str.len()
word_lengths = df_pattern['thought'].str.split().str.len()
print(f'\nThought length stats (characters): '
      f'Mean: {lengths.mean():.0f} | Median: {lengths.median():.0f} | '
      f'Min: {lengths.min()} | Max: {lengths.max()}')
print(f'Thought length stats (words):      '
      f'Mean: {word_lengths.mean():.1f} | Median: {word_lengths.median():.0f} | '
      f'Min: {word_lengths.min()} | Max: {word_lengths.max()}')

# ── Annotator agreement analysis ──
# Each record has 5 annotator votes distributed across 12 categories.
# Check how often the assigned (generation) pattern matches the majority vote.
agree_count = 0
disagree_examples = []
for r in all_records:
    marked = r.get('marked_patterns', {})
    if marked:
        majority_label = max(marked, key=lambda k: int(marked[k]))
        majority_votes = int(marked[majority_label])
        assigned_votes = int(marked.get(r['pattern'], '0'))
        if majority_label == r['pattern']:
            agree_count += 1
        elif len(disagree_examples) < 5:
            disagree_examples.append({
                'thought': r['thought'][:80],
                'assigned': r['pattern'],
                'assigned_votes': assigned_votes,
                'majority': majority_label,
                'majority_votes': majority_votes,
            })

n = len(all_records)
print(f'\n--- Annotator agreement with assigned label ---')
print(f'Assigned label = majority vote: {agree_count}/{n} ({agree_count/n*100:.1f}%)')
print(f'Assigned label ≠ majority vote: {n - agree_count}/{n} ({(n - agree_count)/n*100:.1f}%)')

if disagree_examples:
    print(f'\nExamples where assigned label differs from majority vote:')
    for ex in disagree_examples[:3]:
        print(f'  "{ex["thought"]}..."')
        print(f'    Assigned: {ex["assigned"]} ({ex["assigned_votes"]} votes) | '
              f'Majority: {ex["majority"]} ({ex["majority_votes"]} votes)')

# ── Reframe count statistics ──
reframe_counts = df_pattern['n_reframes'].value_counts().sort_index()
print(f'\nReframes per thought:')
for nref, count in reframe_counts.items():
    print(f'  {nref} reframes: {count} thoughts')
print(f'  Mean reframes per thought: {df_pattern["n_reframes"].mean():.2f}')

# ── Strategy distribution across reframes ──
strategy_counts = {}
for r in all_records:
    for ref in r.get('reframes', []):
        for s in ref.get('original_strategy', []):
            strategy_counts[s] = strategy_counts.get(s, 0) + 1

print(f'\nReframing strategy distribution (across all reframes):')
for strategy, count in sorted(strategy_counts.items(), key=lambda x: -x[1]):
    print(f'  {strategy:25s} {count:5d}')

# ── Sample records ──
print(f'\nSample records:')
for _, row in df_pattern.head(5).iterrows():
    print(f'  [{row["split"]}] [{row["pattern"]}] '
          f'(votes: {row["dominant_votes"]}/{row["total_votes"]}) '
          f'{row["thought"][:100]}')

  reframe_thoughts_dataset_parlai_test.txt: 6807 records  (split="test")
  reframe_thoughts_dataset_parlai_train.txt: 1920 records  (split="train")
  reframe_thoughts_dataset_parlai_valid.txt: 961 records  (split="valid")

Total records across all splits: 9688

DATASET: PatternReframe (Maddela et al., 2023)

Shape: 9688 thoughts, 26507 total reframes
Unique thoughts: 9682
Unique personas: 1158
Splits loaded: test (6807), train (1920), valid (961)

Per-split breakdown:
  split        thoughts   reframes  avg rfr/tht
  test             6807      18635         2.74
  train            1920       5249         2.73
  valid             961       2623         2.73

Label distribution ("pattern") — overall:
  Catastrophizing                                          1027  ( 10.6%)
  Jumping to conclusions: Fortune-telling                  1012  ( 10.4%)
  Jumping to conclusions: mind reading                      992  ( 10.2%)
  Overgeneralization                                        984  ( 10.

---
## 3.1.1 Label Taxonomy Comparison

A cross-dataset comparison of all label strings, case-insensitive. This table reveals naming inconsistencies that must be resolved before merging.

In [32]:
# Collect all unique label strings from each dataset
labels_tqa = sorted(df_therapistqa['Dominant Distortion'].unique().tolist())
labels_dan = sorted(df_danthareja[label_col_d].unique().tolist()) if 'df_danthareja' in dir() else ['(not loaded)']
labels_men = sorted(df_mendeley['FIRST ANNOTATOR'].unique().tolist())
labels_trp = sorted(df_traps['completion'].unique().tolist())
labels_ref = sorted(df_reframing['thinking_traps_addressed'].str.split(',').explode().str.strip().unique().tolist())

all_labels = {
    'TherapistQA': labels_tqa,
    'Danthareja': labels_dan,
    'Mendeley': labels_men,
    'Thinking Traps': labels_trp,
    'Reframing': labels_ref,
}

all_unique = sorted(set(l.lower().strip() for labels in all_labels.values() for l in labels))

# Build comparison table
rows = []
for label in all_unique:
    row = {'Label (normalised)': label}
    for d_name, d_labels in all_labels.items():
        matches = [l for l in d_labels if l.lower().strip() == label]
        row[d_name] = matches[0] if matches else '\u2014'
    rows.append(row)

df_comparison = pd.DataFrame(rows)
display(df_comparison)

,Label (normalised),TherapistQA,Danthareja,Mendeley,Thinking Traps,Reframing
0,all-or-nothing,—,—,All-or-nothing,—,—
1,all-or-nothing thinking,All-or-nothing thinking,All-or-nothing thinking,—,All-or-nothing thinking,all-or-nothing thinking
2,blaming,—,—,—,Blaming,blaming
3,catastrophizing,—,—,—,Catastrophizing,catastrophizing
4,comparing,—,—,—,Comparing,—
5,comparing and despairing,—,—,—,Comparing and Despairing,comparing and despairing
6,discounting the positives,—,—,Discounting the positives,—,—
7,disqualifying the positive,—,—,—,Disqualifying the Positive,disqualifying the positive
8,emotional reasoning,Emotional Reasoning,Emotional Reasoning,Emotional Reasoning,Emotional reasoning,emotional reasoning
9,fortune telling,—,—,—,Fortune telling,fortune telling


---
## 3.1.2 Summary of Critical Findings

### TherapistQA vs Danthareja relationship

The comparison cell above determines the exact relationship between these two datasets. Based on the available_datasets_review.pdf (Chapter 3), the Danthareja dataset is expected to be a superset of TherapistQA with added No Distortion records, but this must be verified from the actual data.

### Cross-dataset overlap

The overlap between Thinking Traps and Cognitive Reframing (both Sharma et al., 2023) is minimal (expected ~9 texts based on preliminary analysis) and will be handled by deduplication on the merged dataset.

### Label taxonomy issues requiring resolution

**1. Name variants of the same concept (must be unified):**
- `Overgeneralization` / `Overgeneralizing` (both present within Thinking Traps)
- `Personalization` / `Personalizing` / `Personalization and Blame` (Mendeley merges two concepts)
- `Disqualifying the Positive` / `Disqualifying the positive` / `Discounting the positives`
- `Fortune-telling` / `Fortune telling` (hyphen difference)
- `Should statements` / `Should statement` (plural/singular)
- `All-or-nothing thinking` / `All-or-nothing`
- `Magnification` / `Magnification or Minimization`
- `No Distortion` / `Not distorted` / `none`

**2. Labels present only in Sharma datasets (not in Burns taxonomy):**
- `Comparing and Despairing` / `Comparing`
- `Blaming`
- `Negative feeling or emotion`
- `Catastrophizing` (Sharma) — may overlap with `Magnification` (Burns)

**3. Structural issues:**
- Thinking Traps has 379 multi-label rows (same prompt, different label rows) — requires decision on handling
- Reframing has 33% multi-label records (comma-separated) and only 286 unique thoughts
- Mendeley has 100% inter-annotator agreement (likely reconciled, not independent)
- Mendeley has 330 augmented records with potentially lower quality
- Mendeley English texts are machine-translated from Indonesian, not native English

In [ ]:
# Final record count summary
print('DATASET SIZE SUMMARY')
print('=' * 60)
n_ori = len(df_mendeley[df_mendeley['DATA STATUS']=='RAW-ORI'])
n_aug = len(df_mendeley[df_mendeley['DATA STATUS']!='RAW-ORI'])
print(f'{"TherapistQA (Kaggle)":40s} {len(df_therapistqa):6d} records')
try:
    print(f'{"Danthareja (Hugging Face)":40s} {len(df_danthareja):6d} records')
except NameError:
    print(f'{"Danthareja (Hugging Face)":40s}   (not loaded)')
print(f'{"Mendeley Indonesian (English)":40s} {len(df_mendeley):6d} records')
print(f'{"  - Original (RAW-ORI)":40s} {n_ori:6d}')
print(f'{"  - Augmented":40s} {n_aug:6d}')
print(f'{"Thinking Traps":40s} {len(df_traps):6d} records ({prompts_clean.nunique()} unique texts)')
print(f'{"Reframing (thought + trap only)":40s} {len(df_reframing):6d} records ({df_reframing["thought"].nunique()} unique texts)')

---
*Step 1 complete. The next cells will proceed with Step 3 (unified label taxonomy design) and Step 4 (label harmonisation).*